# Cloud Provider Analytics — MVP (Segundo Parcial)
**Big Data 72.80 · ITBA · Arquitectura Lambda · PySpark + Structured Streaming + Cassandra/AstraDB**

Integrantes: Camila Lee (63382) · Lucas Perri (62746)

Pipeline end-to-end mínimo: `Landing → Bronze → Silver → Gold → Serving (Cassandra)`, ejecutable de principio a fin en Google Colab.

### Checklist del MVP
- [ ] Batch a Bronze: 3 maestros, Parquet particionado, `ingest_ts`+`source_file`, dedupe
- [ ] Streaming a Bronze: esquema explícito, `withWatermark`, dedupe por `event_id`, late data, checkpoint
- [ ] Silver: limpieza + enriquecimiento + ≥3 features + 3 reglas de calidad + quarantine + flag de anomalía
- [ ] Gold: `org_daily_usage_by_service` (mart FinOps, grano diario por org/servicio)
- [ ] Serving: tabla **CQL** modelada query-first (Cassandra real), cargada desde Spark
- [ ] Q1 + Q2 con resultados (CQL adjunto a cada una)
- [ ] Idempotencia: re-ejecución sin duplicar (conteos antes/después)

### Decisiones de diseño que justificamos en el video
- **Tablas CQL reales** con `PRIMARY KEY ((org_id, service), usage_date)` — modelado *query-first* con clave de partición y de clustering explícitas, **no** colecciones del Document API.
- **Features derivadas de `metric`**: `requests` = `value` cuando `metric='requests'` (contar filas daría un número incorrecto).
- **`value` leído como string y luego casteado** — llega como número, como `"100.0"` o nulo.
- **Anomalías por servicio, marcadas sólo cuando coinciden ≥2 de 3 métodos** — evita el ruido de marcar la mitad de las filas.
- **Costos reales preservados** (marcamos, no recortamos).

## 0. Setup — AstraDB y credenciales (hacer una sola vez)
*Por qué:* Cassandra es la capa de serving; AstraDB es Cassandra gestionado. La conexión usa un **Secure Connect Bundle (SCB)** + un **application token**.

1. En astra.datastax.com creá una base **Serverless (Non-Vector)** y un keyspace **`cloud_analytics`** desde la UI (en serverless no se puede `CREATE KEYSPACE` desde el driver; sí `CREATE TABLE`).
2. Generá un **application token** (rol *Database Administrator*) → copiá el string `AstraCS:...`.
3. Descargá el **Secure Connect Bundle** (zip) y subilo a `/content/` (panel **Archivos → Subir**).
4. Subí el dataset para tener `/content/datalake/landing/` con los 7 CSV y `usage_events_stream/*.jsonl`.

### Credenciales por variables de entorno (no hardcodear)
Las credenciales se leen de **Colab Secrets** o de un archivo **`.env`** (ver `.env.example`). **Nunca** se escriben en el código ni se suben al repo (el `.gitignore` excluye `.env` y el `secure-connect-*.zip`). `/content` se borra entre sesiones; para persistir montá Drive.

In [ ]:
# --- 1. Instalación de dependencias ---
!pip -q install pyspark==3.5.1 cassandra-driver python-dotenv
print("instalado")

In [ ]:
# --- 2. Sesión de Spark ---
# Teoría (Clase Apache Spark — Mosquera): Spark "no reemplaza a Python/Pandas; lo complementa
# cuando la escala o la distribución importan". master("local[*]") usa todos los núcleos de la VM;
# la misma API/plan corre igual en un cluster (lo que respalda nuestra asunción de portabilidad).
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, IntegerType, BooleanType, DateType)
spark = (SparkSession.builder.appName("cloud_provider_analytics_mvp")
         .master("local[*]").config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# --- 3. Rutas y zonas del Data Lake ---
import os
BASE="/content/datalake"; LANDING=f"{BASE}/landing"
BRONZE=f"{BASE}/bronze"; SILVER=f"{BASE}/silver"; GOLD=f"{BASE}/gold"
QUARANTINE=f"{BASE}/quarantine"; CHK="/content/checkpoints"
for p in [BRONZE,SILVER,GOLD,QUARANTINE,CHK]: os.makedirs(p, exist_ok=True)
EVENTS_DIR=f"{LANDING}/usage_events_stream"
print("landing:", os.path.exists(LANDING), "| events:", os.path.exists(EVENTS_DIR))

## 1. Esquemas explícitos
*Teoría (Clase Apache Spark — Mosquera):* "Definir schema explícito cuando sea posible. Evitar `inferSchema` en pipelines productivos." Esto es clave porque el dato es sucio: el esquema de eventos es una **unión nullable de v1+v2** (`carbon_kg`/`genai_tokens` opcionales, sin código condicional), `value` se declara **string** (llega como número, como `"100.0"` o nulo → se castea con fallback) y el campo temporal se llama literalmente `timestamp`.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()), StructField("value", StringType()),   # string -> cast
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()), StructField("genai_tokens", LongType()),  # v2 nullable
])
orgs_schema = StructType([
    StructField("org_id",StringType()),StructField("org_name",StringType()),
    StructField("industry",StringType()),StructField("hq_region",StringType()),
    StructField("plan_tier",StringType()),StructField("is_enterprise",BooleanType()),
    StructField("signup_date",DateType()),StructField("sales_rep",StringType()),
    StructField("lifecycle_stage",StringType()),StructField("marketing_source",StringType()),
    StructField("nps_score",DoubleType())])
users_schema = StructType([
    StructField("user_id",StringType()),StructField("org_id",StringType()),
    StructField("email",StringType()),StructField("role",StringType()),
    StructField("active",BooleanType()),StructField("created_at",DateType()),
    StructField("last_login",DateType())])
billing_schema = StructType([
    StructField("invoice_id",StringType()),StructField("org_id",StringType()),
    StructField("month",DateType()),StructField("subtotal",DoubleType()),
    StructField("credits",DoubleType()),StructField("taxes",DoubleType()),
    StructField("currency",StringType()),StructField("exchange_rate_to_usd",DoubleType())])
print("esquemas listos")

## 2. Batch a Bronze
*Teoría (Clase Apache Spark — Mosquera):* Parquet es "columnar y eficiente… preferido para analytics y data lake". Bronze conserva el mismo grano que la fuente pero **tipado**, **deduplicado** y con columnas técnicas de procedencia (`ingest_ts`, `source_file`), escrito en Parquet particionado.

In [ ]:
def csv_a_bronze(path, schema, key, out):
    df = (spark.read.option("header",True).schema(schema).csv(path)
          .withColumn("ingest_ts",F.current_timestamp())
          .withColumn("source_file",F.input_file_name())
          .dropDuplicates([key]))
    df.write.mode("overwrite").parquet(out); print(f"{out}: {df.count()} filas"); return df

orgs    = csv_a_bronze(f"{LANDING}/customers_orgs.csv", orgs_schema,   "org_id",    f"{BRONZE}/orgs")
users   = csv_a_bronze(f"{LANDING}/users.csv",          users_schema,  "user_id",   f"{BRONZE}/users")
billing = csv_a_bronze(f"{LANDING}/billing_monthly.csv",billing_schema,"invoice_id",f"{BRONZE}/billing")

## 3. Streaming a Bronze (ingesta de la Speed Layer)
*Teoría (Clase Spark Structured Streaming — Mosquera):* el modelo mental correcto es **"tabla no acotada + query incremental"**; Structured Streaming "expresa el pipeline como si fuera batch, pero Spark lo ejecuta incrementalmente". Aplicamos `withWatermark` ("watermark = max event time − retraso permitido") y dedupe por `event_id` (operación con estado), con **checkpointing** habilitado.

Disparador **`trigger(availableNow=True)`**: "procesa lo disponible y termina" — corre toda la maquinaria de streaming sobre el set fijo de archivos y se detiene, lo que hace la corrida reproducible y permite seguir con el resto del notebook.

> Antipatrón (misma clase): "Creer que `readStream` ejecuta la query: ejecuta `writeStream.start()`".

In [ ]:
raw = spark.readStream.schema(event_schema).json(EVENTS_DIR)
bronze_stream = (raw
    .withColumn("event_ts",  F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))   # "100.0"/99.0 -> double; basura -> null
    .withColumn("usage_date",F.to_date("event_ts"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("event_ts","10 minutes")
    .dropDuplicates(["event_id"]))
q = (bronze_stream.writeStream.format("parquet")
     .option("path", f"{BRONZE}/events")
     .option("checkpointLocation", f"{CHK}/bronze_events")
     .outputMode("append").trigger(availableNow=True).start())
q.awaitTermination()
print("streaming -> bronze OK")

In [ ]:
bronze_events = spark.read.parquet(f"{BRONZE}/events")
print("eventos en bronze:", bronze_events.count())
bronze_events.select("event_id","event_ts","org_id","service","metric","value","value_num",
                     "cost_usd_increment","schema_version","genai_tokens").show(5, truncate=False)

## 4. Silver — calidad + quarantine + enriquecimiento
3 reglas de calidad; las filas que fallan van a **quarantine** (se inspeccionan, no se descartan). Además, un `dropDuplicates(["event_id"])` por lotes como red de seguridad global (la deduplicación en streaming sólo cubre la ventana del watermark).

Reglas: `event_id` no nulo · `cost_usd_increment ≥ -0.01` · `unit` no nulo cuando `value` existe.

In [ ]:
ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
valida = (F.col("event_id").isNotNull()
          & (F.col("cost_usd_increment") >= -0.01)
          & ~(F.col("value").isNotNull() & F.col("unit").isNull()))   # regla 3
silver_ok  = ev.filter(valida)
quarantine = ev.filter(~valida)
quarantine.write.mode("overwrite").parquet(f"{QUARANTINE}/events")
print("válidas:", silver_ok.count(), "| en quarantine:", quarantine.count())
quarantine.select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)

### Enriquecimiento (stream-static join)
*Teoría (Clase Spark Structured Streaming — Mosquera):* el **join stream-static** sirve para "enriquecimiento" con una tabla maestra/dimensión. *Clase Apache Spark — Mosquera:* "Si [la tabla] es chica, Spark puede enviarla a cada executor y evitar gran parte del shuffle" → usamos `broadcast` sobre la dimensión de organizaciones.

In [ ]:
orgs_dim = spark.read.parquet(f"{BRONZE}/orgs").select(
    "org_id","industry","plan_tier","hq_region","lifecycle_stage")
silver = silver_ok.join(F.broadcast(orgs_dim), on="org_id", how="left")
print("silver enriquecido:", silver.count())

### 4b. Detección de anomalías — z-score / MAD / p99, flag con ≥2
La consigna pide los tres métodos. Calculamos los estadísticos **por servicio** (los costos tienen escalas distintas por servicio) y marcamos anomalía sólo cuando **coinciden ≥2 de 3** → reduce falsos positivos. Constante `1.4826`: lleva el MAD a la misma escala que el desvío estándar. `K=1.5`: factor sobre p99 (umbral decidido por nosotros — ver log de decisiones). **Marcamos, no recortamos**: las anomalías son la señal de FinOps, no datos a eliminar.

In [ ]:
stats = silver.groupBy("service").agg(
    F.mean("cost_usd_increment").alias("mu"), F.stddev("cost_usd_increment").alias("sd"),
    F.expr("percentile_approx(cost_usd_increment,0.5)").alias("med"),
    F.expr("percentile_approx(cost_usd_increment,0.99)").alias("p99"))
s1 = silver.join(F.broadcast(stats), on="service", how="left") \
           .withColumn("abs_dev", F.abs(F.col("cost_usd_increment")-F.col("med")))
mad = s1.groupBy("service").agg(F.expr("percentile_approx(abs_dev,0.5)").alias("mad"))
s2  = s1.join(F.broadcast(mad), on="service", how="left")

K = 1.5  # factor p99 (decisión de umbral -> log de decisiones)
flag_z   = (F.abs(F.col("cost_usd_increment")-F.col("mu"))/F.col("sd")) > 3
flag_mad = (F.col("mad")>0) & ((F.col("abs_dev")/(1.4826*F.col("mad"))) > 3)
flag_p99 = F.col("cost_usd_increment") > (F.col("p99")*K)
silver_flagged = s2.withColumn("anomaly",
    (flag_z.cast("int")+flag_mad.cast("int")+flag_p99.cast("int")) >= 2)

silver_flagged.write.mode("overwrite").partitionBy("usage_date","service").parquet(f"{SILVER}/events")
print("anomalías marcadas:", silver_flagged.filter("anomaly").count())
silver_flagged.filter("anomaly").select(
    "event_id","org_id","service","cost_usd_increment","mu","p99").orderBy(
    F.desc("cost_usd_increment")).show(10, truncate=False)
silver_flagged.filter("anomaly").groupBy("service").count().show()

## 5. Gold — mart FinOps diario
Grano org×servicio×día. **Features derivadas de `metric`** (no un `count(*)` de filas). Redondeo aplicado en el borde de serving (los decimales largos de `sum()` sobre `double` son artefactos de punto flotante; el dinero se redondea a centavos).

In [ ]:
se = spark.read.parquet(f"{SILVER}/events")
gold_finops = (se.groupBy("org_id","service","usage_date").agg(
        F.round(F.sum("cost_usd_increment"),2).alias("daily_cost_usd"),
        F.round(F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))),2).alias("requests"),
        F.round(F.sum(F.when(F.col("metric")=="cpu_hours", F.col("value_num"))),4).alias("cpu_hours"),
        F.round(F.sum(F.when(F.col("metric")=="storage_gb_hours", F.col("value_num"))),4).alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.round(F.sum("carbon_kg"),6).alias("carbon_kg"))
    .fillna(0, subset=["requests","cpu_hours","storage_gb_hours","genai_tokens","carbon_kg"]))
gold_finops.write.mode("overwrite").partitionBy("usage_date").parquet(f"{GOLD}/org_daily_usage_by_service")
print("filas en gold:", gold_finops.count())
gold_finops.orderBy(F.desc("daily_cost_usd")).show(8, truncate=False)

## 6. Serving — Cassandra/AstraDB (tabla CQL)
Modelado **query-first**: `PRIMARY KEY ((org_id, service), usage_date)` — la clave de partición `(org_id, service)` siempre va en el `WHERE`, y `usage_date` (clustering) permite el rango y el orden. La escritura es **UPSERT** (idempotente).

Credenciales: se leen de **Colab Secrets** o `.env` (nunca hardcodeadas). `dict_factory` hace que las filas se indexen como diccionarios (`r["col"]`).

In [ ]:
# Carga de credenciales: Colab Secrets -> .env -> os.environ (en ese orden de prioridad)
import os
try:
    from dotenv import load_dotenv; load_dotenv()  # lee un archivo .env si existe
except Exception:
    pass
def get_secret(clave, default=None):
    try:
        from google.colab import userdata
        v = userdata.get(clave)
        if v: return v
    except Exception:
        pass
    return os.getenv(clave, default)

SCB_PATH    = get_secret("SCB_PATH", "/content/secure-connect-YOURDB.zip")
ASTRA_TOKEN = get_secret("ASTRA_TOKEN")            # definir en Secrets/.env (AstraCS:...)
KEYSPACE    = get_secret("ASTRA_KEYSPACE", "cloud_analytics")
assert ASTRA_TOKEN, "Falta ASTRA_TOKEN: cargalo en Colab Secrets o en .env (ver .env.example)"
print("credenciales cargadas | keyspace:", KEYSPACE)

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import dict_factory
cluster = Cluster(cloud={"secure_connect_bundle": SCB_PATH},
                  auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN))
session = cluster.connect(KEYSPACE)
session.row_factory = dict_factory
print("conectado:", session.execute("select release_version from system.local").one())

In [ ]:
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_by_service (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
  genai_tokens bigint, carbon_kg double,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')
print("tabla lista")

In [ ]:
import datetime
upsert = session.prepare(f'''
INSERT INTO {KEYSPACE}.org_daily_usage_by_service
(org_id, service, usage_date, daily_cost_usd, requests, cpu_hours, storage_gb_hours, genai_tokens, carbon_kg)
VALUES (?,?,?,?,?,?,?,?,?)''')

def cargar_finops():
    n=0
    for r in gold_finops.toLocalIterator():
        ud=r["usage_date"]
        if isinstance(ud, datetime.datetime): ud=ud.date()
        session.execute(upsert,(r["org_id"], r["service"], ud,
            float(r["daily_cost_usd"] or 0), float(r["requests"] or 0),
            float(r["cpu_hours"] or 0), float(r["storage_gb_hours"] or 0),
            int(r["genai_tokens"] or 0), float(r["carbon_kg"] or 0)))
        n+=1
    return n
print("filas upserteadas:", cargar_finops())

## 7. Consultas Q1 y Q2
Los rangos de fechas se derivan **del propio dato** (sin año hardcodeado, para evitar el desfasaje 2025-vs-2026). Cada consulta imprime su CQL junto al resultado.

- **Q1**: costos y requests diarios por org+servicio en un rango (lectura de una sola partición).
- **Q2**: top-N servicios por costo acumulado en los últimos 14 días para una org. Como el top-N es sobre un valor calculado que cruza particiones, se resuelve leyendo las particiones de la org y agregando del lado de la app (la alternativa de producción es un mart dedicado `top_services_by_org`).

In [ ]:
import pandas as pd, datetime
muestra = gold_finops.orderBy(F.desc("daily_cost_usd")).first()
ORG, SVC = muestra["org_id"], muestra["service"]
limites = gold_finops.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = limites["lo"], limites["hi"]
print(f"Q1 muestra: org={ORG} service={SVC} rango={LO}..{HI}")

q1_cql = f'''SELECT usage_date, daily_cost_usd, requests
FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service='{SVC}'
  AND usage_date>='{LO}' AND usage_date<='{HI}';'''
print("-- Q1 (CQL)\n"+q1_cql)

filas = session.execute(f'''SELECT usage_date, daily_cost_usd, requests
  FROM {KEYSPACE}.org_daily_usage_by_service
  WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''', (ORG,SVC,LO,HI))
pd.DataFrame(list(filas))

In [ ]:
corte = HI - datetime.timedelta(days=14)
print(f"-- Q2 (CQL, repetida por servicio) ventana {corte}..{HI}")
servicios = sorted({r["service"] for r in session.execute(
    f"SELECT org_id, service FROM {KEYSPACE}.org_daily_usage_by_service") if r["org_id"]==ORG})
totales=[]
for s in servicios:
    tot = sum((r["daily_cost_usd"] or 0) for r in session.execute(
        f'''SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
            WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''',
        (ORG, s, corte, HI)))
    totales.append((s, round(tot,4)))
pd.DataFrame(sorted(totales, key=lambda x:-x[1]), columns=["service","cost_14d"])

## 8. Idempotencia
*Teoría (Clase Spark Structured Streaming — Mosquera):* "Fuente replayable + checkpoint + sink idempotente/transaccional = base para semántica exactly-once end-to-end". Cumplimos los tres: archivos (replayables) + checkpoint + UPSERT por clave natural. Re-ejecutar la carga no cambia el conteo.

In [ ]:
antes = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
cargar_finops()
despues  = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
print(f"antes={antes} despues={despues} -> idempotente: {antes==despues}")

## 9. Speed Layer — agregación en streaming → Cassandra con `foreachBatch`
Esta es la versión *streaming* de la capa de velocidad: una agregación por **ventana diaria (tumbling)** sobre event-time, escrita a Cassandra en cada micro-batch con **`foreachBatch`**.

*Teoría (Clase Spark Structured Streaming — Mosquera):* `foreachBatch` permite "escrituras idempotentes por micro-batch" hacia sinks que el streaming no soporta nativamente (como AstraDB). Usamos **output mode `update`** porque, según la clase, "si el resultado todavía puede cambiar por eventos tardíos, `append` sólo es seguro… [con] una ventana cerrada por watermark" — una agregación que se actualiza va con `update`, y el UPSERT de Cassandra sobreescribe por clave. Disparador `availableNow` para una corrida reproducible.

In [ ]:
# Speed Layer: agregación en streaming -> Cassandra vía foreachBatch
stream_silver = (spark.readStream.schema(event_schema).json(EVENTS_DIR)
    .withColumn("event_ts", F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))
    .filter((F.col("event_id").isNotNull()) & (F.col("cost_usd_increment") >= -0.01))
    .withWatermark("event_ts", "10 minutes"))

stream_gold = (stream_silver
    .groupBy(F.window("event_ts", "1 day"), "org_id", "service")   # ventana tumbling diaria
    .agg(F.sum("cost_usd_increment").alias("daily_cost_usd"),
         F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))).alias("requests"))
    .withColumn("usage_date", F.to_date(F.col("window.start"))))

def a_cassandra(batch_df, batch_id):
    for r in batch_df.collect():
        session.execute(upsert, (r["org_id"], r["service"], r["usage_date"],
            float(r["daily_cost_usd"] or 0), float(r["requests"] or 0), 0.0, 0.0, 0, 0.0))

qs = (stream_gold.writeStream
      .outputMode("update")                 # agregación que cambia -> update
      .foreachBatch(a_cassandra)
      .option("checkpointLocation", f"{CHK}/speed_finops")
      .trigger(availableNow=True)
      .start())
qs.awaitTermination()
print("speed layer OK:", qs.lastProgress is not None)

## 10. Observabilidad y cierre
*Teoría (Clase Spark Structured Streaming — Mosquera):* "un pipeline streaming sin monitoreo es una deuda operativa: puede estar corriendo y estar atrasado". Inspeccionamos `lastProgress` y cerramos las queries con `query.stop()` (antipatrón: dejar queries infinitas en el notebook).

In [ ]:
print("lastProgress (ingesta):", q.lastProgress)
for s in spark.streams.active: s.stop()
print("streams activos:", spark.streams.active)